# Week 12: Bayesian Statistics — Applied Statistics with Python (2026)

This week we explore **Bayesian statistics**, a fundamentally different approach to statistical inference. Instead of treating parameters as fixed unknowns, Bayesian methods treat them as random variables with probability distributions — allowing us to update our beliefs as new data arrives.

| # | Topic |
|---|-------|
| 1 | Bayesian vs Frequentist Thinking |
| 2 | Bayes' Theorem Foundations |
| 3 | Prior Distributions |
| 4 | Likelihood and Posterior |
| 5 | Conjugate Priors: Beta-Binomial |
| 6 | Conjugate Priors: Normal-Normal |
| 7 | Credible Intervals vs Confidence Intervals |
| 8 | Maximum A Posteriori (MAP) Estimation |
| 9 | Introduction to MCMC |
| 10 | Bayesian Analysis with PyMC |
| 11 | Practical Example: Bayesian A/B Testing |
| 12 | Summary & Homework |

## Imports

We import all necessary libraries at the top. This week we use `scipy.stats` for distributions and optionally `pymc` for probabilistic programming.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.special import beta as beta_func
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

# Plot settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
np.random.seed(42)

print('All imports successful!')

---
## 1. Bayesian vs Frequentist Thinking

Statistics has two major schools of thought:

| Aspect | Frequentist | Bayesian |
|--------|-------------|----------|
| **Parameters** | Fixed but unknown constants | Random variables with distributions |
| **Probability** | Long-run frequency of events | Degree of belief / uncertainty |
| **Data** | Random (from repeated sampling) | Fixed (what we observed) |
| **Inference** | P(data \| parameter) | P(parameter \| data) |
| **Prior knowledge** | Not formally used | Encoded as prior distribution |
| **Result** | Point estimate + confidence interval | Full posterior distribution |

**Key Bayesian idea:** We start with a *prior belief* about a parameter, observe data, and update to a *posterior belief* using Bayes' theorem.

### 1.1 A Simple Intuition: The Coin Example

Let's compare how frequentists and Bayesians think about a possibly unfair coin. We flip it 10 times and get 7 heads.

In [ ]:
# Frequentist approach: point estimate + CI
n_flips = 10
n_heads = 7
p_hat = n_heads / n_flips  # MLE estimate

# Wilson confidence interval (better for small n)
from statsmodels.stats.proportion import proportion_confint
ci_low, ci_high = proportion_confint(n_heads, n_flips, alpha=0.05, method='wilson')

print('=== Frequentist Approach ===')
print(f'Point estimate (MLE): p = {p_hat:.3f}')
print(f'95% CI: ({ci_low:.3f}, {ci_high:.3f})')
print(f'Interpretation: If we repeated this experiment many times,')
print(f'95% of such intervals would contain the true p.')

print()

# Bayesian approach: start with prior, compute posterior
# Prior: Beta(1,1) = Uniform (no prior preference)
alpha_prior, beta_prior = 1, 1  # uniform prior
alpha_post = alpha_prior + n_heads       # posterior alpha
beta_post = beta_prior + (n_flips - n_heads)  # posterior beta

posterior = stats.beta(alpha_post, beta_post)
post_mean = posterior.mean()
post_ci = posterior.interval(0.95)  # 95% credible interval

print('=== Bayesian Approach ===')
print(f'Prior: Beta({alpha_prior}, {beta_prior}) — uniform, no prior info')
print(f'Posterior: Beta({alpha_post}, {beta_post})')
print(f'Posterior mean: p = {post_mean:.3f}')
print(f'95% Credible Interval: ({post_ci[0]:.3f}, {post_ci[1]:.3f})')
print(f'Interpretation: There is a 95% probability that p lies in this interval.')

### 1.2 Visualizing the Difference

The Bayesian approach gives us a full distribution over the parameter, not just a point estimate. Let's visualize the prior, likelihood, and posterior.

In [ ]:
x = np.linspace(0, 1, 500)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Prior
prior = stats.beta(alpha_prior, beta_prior)
axes[0].plot(x, prior.pdf(x), 'b-', lw=2)
axes[0].fill_between(x, prior.pdf(x), alpha=0.2, color='blue')
axes[0].set_title('Prior: Beta(1,1)')
axes[0].set_xlabel('p (probability of heads)')
axes[0].set_ylabel('Density')

# Likelihood (proportional to Binomial)
likelihood = x**n_heads * (1-x)**(n_flips - n_heads)
likelihood = likelihood / likelihood.max()  # normalize for plotting
axes[1].plot(x, likelihood, 'g-', lw=2)
axes[1].fill_between(x, likelihood, alpha=0.2, color='green')
axes[1].set_title(f'Likelihood: {n_heads} heads in {n_flips} flips')
axes[1].set_xlabel('p (probability of heads)')

# Posterior
posterior = stats.beta(alpha_post, beta_post)
axes[2].plot(x, posterior.pdf(x), 'r-', lw=2)
axes[2].fill_between(x, posterior.pdf(x), alpha=0.2, color='red')
axes[2].axvline(post_mean, color='red', ls='--', label=f'Mean={post_mean:.3f}')
axes[2].axvline(p_hat, color='green', ls=':', label=f'MLE={p_hat:.3f}')
axes[2].set_title(f'Posterior: Beta({alpha_post},{beta_post})')
axes[2].set_xlabel('p (probability of heads)')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Prior × Likelihood ∝ Posterior', y=1.02, fontsize=14, fontweight='bold')
plt.show()

---
## 2. Bayes' Theorem Foundations

Bayes' theorem is the mathematical backbone of Bayesian inference:

$$P(\theta | D) = \frac{P(D | \theta) \cdot P(\theta)}{P(D)}$$

Where:
- $P(\theta | D)$ — **Posterior**: updated belief about parameter $\theta$ after seeing data $D$
- $P(D | \theta)$ — **Likelihood**: probability of observing data given $\theta$
- $P(\theta)$ — **Prior**: initial belief about $\theta$ before seeing data
- $P(D)$ — **Evidence** (marginal likelihood): normalizing constant

In practice we often write: **Posterior ∝ Likelihood × Prior** (ignoring the normalizing constant).

### 2.1 Discrete Bayes' Theorem

Let's start with a classic discrete example — a medical screening test.

In [ ]:
# Medical test example
# A disease affects 1% of the population
# Test sensitivity (true positive rate): 95%
# Test specificity (true negative rate): 90%

p_disease = 0.01           # prior: P(disease)
p_no_disease = 1 - p_disease
p_positive_given_disease = 0.95    # sensitivity
p_positive_given_no_disease = 0.10 # false positive rate (1 - specificity)

# P(positive) — total probability
p_positive = (p_positive_given_disease * p_disease + 
              p_positive_given_no_disease * p_no_disease)

# P(disease | positive) — Bayes' theorem
p_disease_given_positive = (p_positive_given_disease * p_disease) / p_positive

print('=== Medical Screening Test ===')
print(f'Prior P(disease)            = {p_disease:.4f} (1%)')
print(f'Sensitivity P(+|disease)    = {p_positive_given_disease:.4f}')
print(f'False positive P(+|healthy) = {p_positive_given_no_disease:.4f}')
print(f'P(positive test)            = {p_positive:.4f}')
print(f'\nPosterior P(disease|+)      = {p_disease_given_positive:.4f} ({p_disease_given_positive*100:.1f}%)')
print(f'\nSurprise! Even with a positive test, there is only a')
print(f'{p_disease_given_positive*100:.1f}% chance of actually having the disease.')
print(f'This is because the disease is rare (low prior).')

### 2.2 Sequential Updating

A key advantage of Bayesian inference is **sequential updating** — the posterior from one observation becomes the prior for the next. Let's see how our belief about the coin updates as we observe more flips.

In [ ]:
# Simulate sequential coin flips with true p = 0.6
true_p = 0.6
np.random.seed(42)
flips = np.random.binomial(1, true_p, size=100)  # 1=heads, 0=tails

# Track posterior after each flip
x = np.linspace(0, 1, 500)
alpha, beta_param = 1, 1  # start with uniform prior

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
checkpoints = [0, 1, 3, 5, 10, 20, 50, 100]  # after N flips

a_track, b_track = 1, 1
for idx, n in enumerate(checkpoints):
    ax = axes[idx // 4][idx % 4]
    
    # Compute posterior after n flips
    heads = flips[:n].sum() if n > 0 else 0
    tails = n - heads
    a_post = 1 + heads  # alpha_prior + heads
    b_post = 1 + tails  # beta_prior + tails
    
    post = stats.beta(a_post, b_post)
    ax.plot(x, post.pdf(x), 'b-', lw=2)
    ax.fill_between(x, post.pdf(x), alpha=0.2)
    ax.axvline(true_p, color='red', ls='--', lw=1.5, label=f'True p={true_p}')
    ax.set_title(f'After {n} flips\nBeta({a_post},{b_post})', fontsize=10)
    ax.set_xlim(0, 1)
    if idx == 0:
        ax.legend(fontsize=8)

plt.suptitle('Sequential Bayesian Updating: Belief About Coin Bias', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('As more data arrives, the posterior concentrates around the true value.')
print('This demonstrates how Bayesian learning works — uncertainty shrinks with data.')

---
## 3. Prior Distributions

The **prior** encodes what we know (or assume) about a parameter *before* seeing data. Choosing a good prior is both an art and a science.

| Prior Type | Description | Example |
|------------|-------------|---------|
| **Informative** | Strong belief based on domain knowledge | Beta(100, 100) — coin is likely fair |
| **Weakly informative** | Gentle constraint, rules out implausible values | Beta(2, 2) — slight preference for middle |
| **Non-informative (flat)** | Minimal assumption, let data speak | Beta(1, 1) = Uniform |
| **Jeffreys** | Invariant under reparameterization | Beta(0.5, 0.5) for binomial |

### 3.1 Impact of Different Priors

Let's see how different priors affect the posterior when we observe the same data (7 heads in 10 flips).

In [ ]:
n_flips, n_heads = 10, 7
x = np.linspace(0, 1, 500)

# Define different priors
priors = {
    'Flat: Beta(1,1)': (1, 1),
    'Jeffreys: Beta(0.5,0.5)': (0.5, 0.5),
    'Weakly informative: Beta(2,2)': (2, 2),
    'Informative (fair): Beta(20,20)': (20, 20),
    'Strong (biased high): Beta(30,10)': (30, 10),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors = plt.cm.Set1(np.linspace(0, 1, len(priors)))

for i, (name, (a, b)) in enumerate(priors.items()):
    # Plot prior
    prior_dist = stats.beta(a, b)
    axes[0].plot(x, prior_dist.pdf(x), color=colors[i], lw=2, label=name)
    
    # Compute and plot posterior
    a_post = a + n_heads
    b_post = b + (n_flips - n_heads)
    post_dist = stats.beta(a_post, b_post)
    axes[1].plot(x, post_dist.pdf(x), color=colors[i], lw=2,
                 label=f'Post Beta({a_post},{b_post}), mean={post_dist.mean():.3f}')

axes[0].set_title('Prior Distributions', fontsize=13)
axes[0].set_xlabel('p')
axes[0].legend(fontsize=9)

axes[1].set_title(f'Posterior after {n_heads}/{n_flips} heads', fontsize=13)
axes[1].set_xlabel('p')
axes[1].axvline(0.7, color='black', ls=':', label='MLE=0.7')
axes[1].legend(fontsize=9)

plt.suptitle('How Prior Choice Affects the Posterior', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key observations:')
print('1. Flat/Jeffreys priors → posterior ≈ driven by data (MLE)')
print('2. Informative prior (fair coin) → pulls posterior toward 0.5')
print('3. With more data, all priors converge to similar posteriors')

### 3.2 Prior Swamping — Large Data Overcomes Priors

With enough data, even a strong prior gets overwhelmed. This is called **prior swamping** — the likelihood dominates.

In [ ]:
# Show how a wrong strong prior gets corrected by data
# True p = 0.7, but strong prior says p ≈ 0.3
true_p = 0.7
strong_a, strong_b = 30, 70  # prior mean = 0.3

sample_sizes = [0, 10, 50, 100, 500, 1000]
x = np.linspace(0, 1, 500)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
np.random.seed(42)
all_flips = np.random.binomial(1, true_p, 1000)

for idx, n in enumerate(sample_sizes):
    ax = axes[idx // 3][idx % 3]
    heads = all_flips[:n].sum() if n > 0 else 0
    a_post = strong_a + heads
    b_post = strong_b + (n - heads)
    post = stats.beta(a_post, b_post)
    
    ax.plot(x, post.pdf(x), 'b-', lw=2)
    ax.fill_between(x, post.pdf(x), alpha=0.2)
    ax.axvline(true_p, color='red', ls='--', lw=2)
    ax.set_title(f'n={n}, mean={post.mean():.3f}', fontsize=11)
    ax.set_xlim(0, 1)

plt.suptitle('Prior Swamping: Strong Wrong Prior (mean=0.3) vs True p=0.7',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Even a strongly wrong prior (Beta(30,70), mean=0.3) gets corrected')
print('by ~500 observations. Data always wins eventually.')

---
## 4. Likelihood and Posterior

The **likelihood function** $L(\theta | D) = P(D | \theta)$ measures how well each parameter value explains the observed data. Combined with the prior, it produces the posterior.

### Computing the Posterior: Grid Approximation

When analytical solutions aren't available, we can approximate the posterior using a **grid of parameter values**. This is the simplest numerical method.

In [ ]:
def grid_posterior(data, prior_func, likelihood_func, grid_size=1000):
    """
    Compute posterior using grid approximation.
    
    Parameters:
    -----------
    data : observed data
    prior_func : function that returns prior density for each grid point
    likelihood_func : function that returns likelihood for each grid point given data
    grid_size : number of grid points
    
    Returns:
    --------
    grid : parameter values
    posterior : normalized posterior probabilities
    """
    grid = np.linspace(0.001, 0.999, grid_size)  # parameter grid
    prior = prior_func(grid)                       # prior at each point
    likelihood = likelihood_func(grid, data)       # likelihood at each point
    unstd_posterior = likelihood * prior            # unnormalized posterior
    posterior = unstd_posterior / unstd_posterior.sum()  # normalize
    return grid, posterior

# Example: Coin flip — 14 heads in 20 flips
n, k = 20, 14
data = {'n': n, 'k': k}

# Define prior and likelihood
prior_func = lambda grid: stats.beta(2, 2).pdf(grid)  # weakly informative
likelihood_func = lambda grid, d: stats.binom.pmf(d['k'], d['n'], grid)

# Compute
grid, posterior = grid_posterior(data, prior_func, likelihood_func)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grid, prior_func(grid) / prior_func(grid).sum(), 'b--', lw=2, label='Prior')
ax.plot(grid, posterior, 'r-', lw=2, label='Posterior')
ax.axvline(k/n, color='green', ls=':', lw=2, label=f'MLE = {k/n:.2f}')
ax.set_xlabel('p (probability of heads)')
ax.set_ylabel('Probability (grid approximation)')
ax.set_title(f'Grid Approximation: {k} heads in {n} flips')
ax.legend()
plt.tight_layout()
plt.show()

# Summary statistics from grid
post_mean = np.sum(grid * posterior)
post_mode = grid[np.argmax(posterior)]
cumsum = np.cumsum(posterior)
ci_low = grid[np.searchsorted(cumsum, 0.025)]
ci_high = grid[np.searchsorted(cumsum, 0.975)]

print(f'Posterior mean: {post_mean:.4f}')
print(f'Posterior mode (MAP): {post_mode:.4f}')
print(f'95% Credible interval: ({ci_low:.4f}, {ci_high:.4f})')

---
## 5. Conjugate Priors: Beta-Binomial Model

A **conjugate prior** is a prior that, when combined with a particular likelihood, produces a posterior of the same family. This gives us analytical (closed-form) solutions.

The most important conjugate pair:

| Likelihood | Conjugate Prior | Posterior |
|------------|----------------|----------|
| Binomial(n, p) | Beta(α, β) | Beta(α + k, β + n − k) |
| Normal(μ, σ² known) | Normal(μ₀, σ₀²) | Normal(μ_post, σ²_post) |
| Poisson(λ) | Gamma(α, β) | Gamma(α + Σx, β + n) |
| Normal(μ known, σ²) | Inverse-Gamma(α, β) | Inverse-Gamma(α + n/2, β + Σ(x-μ)²/2) |

### 5.1 Beta-Binomial: The Complete Workflow

For a binomial likelihood (success/failure data) with a Beta prior, the math is elegant.

In [ ]:
def beta_binomial_analysis(k, n, alpha_prior, beta_prior, name=''):
    """
    Complete Beta-Binomial Bayesian analysis.
    
    Parameters:
    -----------
    k : number of successes
    n : number of trials
    alpha_prior, beta_prior : Beta prior parameters
    name : label for this analysis
    """
    # Posterior parameters
    alpha_post = alpha_prior + k
    beta_post = beta_prior + (n - k)
    
    # Distributions
    prior = stats.beta(alpha_prior, beta_prior)
    posterior = stats.beta(alpha_post, beta_post)
    
    # Summary
    print(f'=== Beta-Binomial Analysis{" — " + name if name else ""} ===')
    print(f'Data: {k} successes in {n} trials')
    print(f'Prior: Beta({alpha_prior}, {beta_prior}), mean = {prior.mean():.4f}')
    print(f'Posterior: Beta({alpha_post}, {beta_post})')
    print(f'  Mean:   {posterior.mean():.4f}')
    print(f'  Median: {posterior.median():.4f}')
    print(f'  Mode:   {(alpha_post-1)/(alpha_post+beta_post-2):.4f}')
    print(f'  Std:    {posterior.std():.4f}')
    ci = posterior.interval(0.95)
    print(f'  95% CI: ({ci[0]:.4f}, {ci[1]:.4f})')
    print(f'  MLE:    {k/n:.4f}')
    
    # Effective sample size of prior
    ess = alpha_prior + beta_prior
    print(f'  Prior effective sample size: {ess}')
    print(f'  Posterior is weighted average: '
          f'{ess/(ess+n):.1%} prior + {n/(ess+n):.1%} data')
    
    return alpha_post, beta_post

# Example: Customer conversion rate
# 45 out of 200 visitors converted
alpha_post, beta_post = beta_binomial_analysis(
    k=45, n=200, alpha_prior=2, beta_prior=8, 
    name='Website Conversion Rate'
)

### 5.2 Visualizing the Beta-Binomial Update

Let's create a comprehensive visualization showing prior, likelihood, and posterior for the conversion rate example.

In [ ]:
k, n = 45, 200
alpha_prior, beta_prior = 2, 8
alpha_post = alpha_prior + k
beta_post = beta_prior + (n - k)

x = np.linspace(0, 0.5, 500)

prior = stats.beta(alpha_prior, beta_prior)
posterior = stats.beta(alpha_post, beta_post)
# Likelihood (scaled for visualization)
log_lik = stats.binom.logpmf(k, n, x)
lik = np.exp(log_lik - log_lik.max())  # normalized to max=1

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, prior.pdf(x), 'b--', lw=2, label=f'Prior: Beta({alpha_prior},{beta_prior})')
ax.plot(x, lik * posterior.pdf(x).max(), 'g:', lw=2, label='Likelihood (scaled)')
ax.plot(x, posterior.pdf(x), 'r-', lw=3, label=f'Posterior: Beta({alpha_post},{beta_post})')

# Shade 95% credible interval
ci = posterior.interval(0.95)
mask = (x >= ci[0]) & (x <= ci[1])
ax.fill_between(x[mask], posterior.pdf(x[mask]), alpha=0.2, color='red',
                label=f'95% CI: ({ci[0]:.3f}, {ci[1]:.3f})')

ax.axvline(k/n, color='green', ls=':', alpha=0.7)
ax.axvline(posterior.mean(), color='red', ls='--', alpha=0.7)

ax.set_xlabel('Conversion Rate (p)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Beta-Binomial: Website Conversion Rate', fontsize=14)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

### 5.3 Bayesian Hypothesis Testing with Beta-Binomial

In Bayesian framework, we can directly compute the probability that a parameter exceeds a threshold — much more intuitive than p-values.

In [ ]:
# Is the conversion rate > 20%?
threshold = 0.20
posterior = stats.beta(alpha_post, beta_post)

# P(p > threshold | data)
prob_above = 1 - posterior.cdf(threshold)

print(f'Question: Is the conversion rate above {threshold:.0%}?')
print(f'P(p > {threshold} | data) = {prob_above:.4f} ({prob_above*100:.1f}%)')
print()

# More thresholds
thresholds = [0.15, 0.20, 0.25, 0.30]
print('Probability that conversion rate exceeds various thresholds:')
print('-' * 40)
for t in thresholds:
    p = 1 - posterior.cdf(t)
    print(f'  P(p > {t:.2f}) = {p:.4f} ({p*100:.1f}%)')

print()
print('Compare to frequentist: we can only reject/fail-to-reject H₀.')
print('Bayesian: we get direct probability statements about the parameter!')

---
## 6. Conjugate Priors: Normal-Normal Model

When data comes from a Normal distribution with **known variance**, and we place a Normal prior on the mean, the posterior is also Normal.

**Setup:**
- Data: $x_1, \ldots, x_n \sim N(\mu, \sigma^2)$ where $\sigma^2$ is known
- Prior: $\mu \sim N(\mu_0, \sigma_0^2)$
- Posterior: $\mu | \text{data} \sim N(\mu_\text{post}, \sigma_\text{post}^2)$

Where:
$$\mu_\text{post} = \frac{\frac{\mu_0}{\sigma_0^2} + \frac{n\bar{x}}{\sigma^2}}{\frac{1}{\sigma_0^2} + \frac{n}{\sigma^2}}, \quad \sigma_\text{post}^2 = \frac{1}{\frac{1}{\sigma_0^2} + \frac{n}{\sigma^2}}$$

### 6.1 Normal-Normal Update

Let's implement the Normal-Normal conjugate update and see how data updates our belief about the mean.

In [ ]:
def normal_normal_update(data, sigma_known, mu_prior, sigma_prior):
    """
    Normal-Normal conjugate update.
    
    Parameters:
    -----------
    data : array of observations
    sigma_known : known population standard deviation
    mu_prior : prior mean
    sigma_prior : prior standard deviation
    
    Returns:
    --------
    mu_post, sigma_post : posterior mean and std
    """
    n = len(data)
    x_bar = np.mean(data)
    
    # Precision (1/variance) formulation
    prec_prior = 1 / sigma_prior**2
    prec_data = n / sigma_known**2
    prec_post = prec_prior + prec_data
    
    # Posterior parameters
    mu_post = (prec_prior * mu_prior + prec_data * x_bar) / prec_post
    sigma_post = np.sqrt(1 / prec_post)
    
    return mu_post, sigma_post

# Example: Estimating average student exam score
# Prior belief: mean ≈ 70, somewhat uncertain (σ₀ = 10)
# Known σ = 15 (population std of exam scores)
mu_prior, sigma_prior = 70, 10
sigma_known = 15

# Observe 25 students with mean = 78
np.random.seed(42)
data = np.random.normal(78, sigma_known, size=25)
x_bar = data.mean()

mu_post, sigma_post = normal_normal_update(data, sigma_known, mu_prior, sigma_prior)

print('=== Normal-Normal Bayesian Update ===')
print(f'Prior: μ ~ N({mu_prior}, {sigma_prior}²)')
print(f'Data: n={len(data)}, x̄={x_bar:.2f}, σ={sigma_known} (known)')
print(f'Posterior: μ ~ N({mu_post:.2f}, {sigma_post:.2f}²)')
print(f'  95% CI: ({mu_post - 1.96*sigma_post:.2f}, {mu_post + 1.96*sigma_post:.2f})')
print(f'\nThe posterior mean ({mu_post:.2f}) is between the prior ({mu_prior}) and data ({x_bar:.2f}).')

### 6.2 Visualizing Normal-Normal Update

Let's plot the prior, likelihood, and posterior to see the precision-weighted averaging.

In [ ]:
x_range = np.linspace(40, 100, 500)

prior_dist = stats.norm(mu_prior, sigma_prior)
lik_sigma = sigma_known / np.sqrt(len(data))  # std of sampling distribution
lik_dist = stats.norm(x_bar, lik_sigma)
post_dist = stats.norm(mu_post, sigma_post)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_range, prior_dist.pdf(x_range), 'b--', lw=2,
        label=f'Prior: N({mu_prior}, {sigma_prior}²)')
ax.plot(x_range, lik_dist.pdf(x_range), 'g:', lw=2,
        label=f'Likelihood: N({x_bar:.1f}, {lik_sigma:.1f}²)')
ax.plot(x_range, post_dist.pdf(x_range), 'r-', lw=3,
        label=f'Posterior: N({mu_post:.1f}, {sigma_post:.1f}²)')

ax.axvline(mu_prior, color='blue', ls='--', alpha=0.3)
ax.axvline(x_bar, color='green', ls=':', alpha=0.3)
ax.axvline(mu_post, color='red', ls='-', alpha=0.3)

ax.set_xlabel('μ (mean exam score)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Normal-Normal Conjugate Update', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# Weight analysis
w_prior = (1/sigma_prior**2) / (1/sigma_prior**2 + len(data)/sigma_known**2)
w_data = 1 - w_prior
print(f'Posterior mean = {w_prior:.1%} × prior + {w_data:.1%} × data')
print(f'  = {w_prior:.3f} × {mu_prior} + {w_data:.3f} × {x_bar:.2f}')
print(f'  = {w_prior * mu_prior + w_data * x_bar:.2f}')
print(f'\nNote: posterior is narrower than both prior and likelihood!')

---
## 7. Credible Intervals vs Confidence Intervals

These two concepts are often confused but have fundamentally different interpretations:

| Feature | Confidence Interval (Frequentist) | Credible Interval (Bayesian) |
|---------|----------------------------------|-----------------------------|
| **Statement** | "95% of CIs from repeated samples contain θ" | "95% probability that θ lies in this interval" |
| **Parameter** | Fixed but unknown | Random variable |
| **Interval** | Random (changes with each sample) | Fixed (given data) |
| **Probability** | About the procedure, not the parameter | About the parameter directly |
| **Types** | One main type | HPD, equal-tailed, etc. |

### 7.1 Highest Posterior Density (HPD) Interval

The **HPD interval** is the shortest interval containing a given probability. For symmetric distributions, it equals the equal-tailed interval; for skewed distributions, it's shorter.

In [ ]:
def hpd_interval(dist, prob=0.95, n_points=10000):
    """
    Compute the Highest Posterior Density (HPD) interval.
    Uses the shortest interval approach.
    """
    # Generate quantile-based grid
    alpha = 1 - prob
    lower_quantiles = np.linspace(0, alpha, n_points)
    upper_quantiles = lower_quantiles + prob
    
    # Find interval bounds
    lowers = dist.ppf(lower_quantiles)
    uppers = dist.ppf(upper_quantiles)
    
    # Find shortest interval
    widths = uppers - lowers
    best = np.argmin(widths)
    
    return lowers[best], uppers[best]

# Compare equal-tailed vs HPD for a skewed posterior
# Beta(3, 10) — skewed right
posterior = stats.beta(3, 10)

# Equal-tailed 95% interval
et_low, et_high = posterior.interval(0.95)

# HPD 95% interval
hpd_low, hpd_high = hpd_interval(posterior, 0.95)

print('Skewed posterior: Beta(3, 10)')
print(f'  Mean = {posterior.mean():.4f}')
print(f'  Mode = {(3-1)/(3+10-2):.4f}')
print()
print(f'Equal-tailed 95% CI: ({et_low:.4f}, {et_high:.4f})  width = {et_high-et_low:.4f}')
print(f'HPD 95% CI:          ({hpd_low:.4f}, {hpd_high:.4f})  width = {hpd_high-hpd_low:.4f}')
print(f'\nHPD is shorter by {(et_high-et_low)-(hpd_high-hpd_low):.4f}')

### 7.2 Visualizing HPD vs Equal-Tailed Intervals

Let's visualize both interval types on a skewed posterior to see the difference.

In [ ]:
x = np.linspace(0, 0.7, 500)
posterior = stats.beta(3, 10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (low, high, title) in zip(axes, [
    (et_low, et_high, 'Equal-Tailed 95% CI'),
    (hpd_low, hpd_high, 'HPD 95% CI (shortest)')
]):
    ax.plot(x, posterior.pdf(x), 'k-', lw=2)
    mask = (x >= low) & (x <= high)
    ax.fill_between(x[mask], posterior.pdf(x[mask]), alpha=0.3, color='red')
    ax.axvline(low, color='red', ls='--', lw=1.5)
    ax.axvline(high, color='red', ls='--', lw=1.5)
    ax.set_title(f'{title}\n({low:.4f}, {high:.4f}), width={high-low:.4f}', fontsize=11)
    ax.set_xlabel('θ')
    ax.set_ylabel('Density')

plt.suptitle('Beta(3,10): HPD vs Equal-Tailed Intervals', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('For skewed distributions, HPD is shorter and all points inside')
print('have higher density than points outside.')

---
## 8. Maximum A Posteriori (MAP) Estimation

The **MAP estimate** is the mode of the posterior distribution — the single most probable parameter value given the data and prior.

$$\hat{\theta}_{MAP} = \arg\max_\theta \, P(\theta | D) = \arg\max_\theta \, P(D|\theta) \cdot P(\theta)$$

| Estimator | Formula | Prior influence |
|-----------|---------|----------------|
| **MLE** | $\arg\max_\theta P(D \| \theta)$ | None — ignores prior |
| **MAP** | $\arg\max_\theta P(D \| \theta) P(\theta)$ | Uses prior as regularizer |
| **Posterior Mean** | $E[\theta \| D]$ | Full Bayesian average |

MAP sits between MLE and the full Bayesian approach. It incorporates prior information but returns a single point estimate rather than a full distribution.

### 8.1 MAP vs MLE Comparison

Let's compare MLE and MAP estimates for the coin flip problem with different priors.

In [ ]:
# Compare MLE, MAP, and posterior mean for Beta-Binomial
n_flips, n_heads = 10, 7

# MLE is always k/n regardless of prior
mle = n_heads / n_flips

print(f'Data: {n_heads} heads in {n_flips} flips')
print(f'MLE: {mle:.4f}\n')
print(f'{"Prior":<30} {"MAP":>8} {"Post Mean":>10} {"MLE":>8}')
print('-' * 60)

priors = {
    'Flat Beta(1,1)': (1, 1),
    'Jeffreys Beta(0.5,0.5)': (0.5, 0.5),
    'Weak Beta(2,2)': (2, 2),
    'Informative Beta(10,10)': (10, 10),
    'Strong fair Beta(50,50)': (50, 50),
}

for name, (a, b) in priors.items():
    a_post = a + n_heads
    b_post = b + (n_flips - n_heads)
    post = stats.beta(a_post, b_post)
    
    # MAP = mode of posterior Beta = (a-1)/(a+b-2) when a,b > 1
    if a_post > 1 and b_post > 1:
        map_est = (a_post - 1) / (a_post + b_post - 2)
    else:
        map_est = float('nan')
    
    post_mean = post.mean()
    print(f'{name:<30} {map_est:>8.4f} {post_mean:>10.4f} {mle:>8.4f}')

print()
print('Key insight: MAP is pulled toward the prior mode.')
print('With flat prior, MAP = MLE. Stronger priors → more shrinkage.')

### 8.2 MAP as Regularized MLE

MAP estimation is mathematically equivalent to **regularized maximum likelihood**. For example, a Normal prior on regression coefficients gives L2 regularization (Ridge regression).

In [ ]:
# MAP as regularization — Normal prior on mean
# log P(θ|D) = log P(D|θ) + log P(θ) + const
# For Normal prior N(0, τ²): log P(θ) = -θ²/(2τ²) + const
# This is exactly L2 regularization!

from scipy.optimize import minimize_scalar

# Observe data from N(μ, σ²=1), find MAP with N(0, τ²) prior
np.random.seed(42)
true_mu = 3.0
data = np.random.normal(true_mu, 1.0, size=5)  # small sample
x_bar = data.mean()

tau_values = [0.5, 1.0, 2.0, 5.0, 100.0]  # prior std (smaller = stronger regularization)
sigma = 1.0
n = len(data)

print(f'Data: n={n}, x̄={x_bar:.3f}, true μ={true_mu}')
print(f'MLE = x̄ = {x_bar:.3f}\n')
print(f'{"Prior N(0,τ²)":<20} {"τ":>5} {"MAP":>8} {"Shrinkage":>10}')
print('-' * 48)

for tau in tau_values:
    # MAP for Normal-Normal: weighted average
    map_est = (n * x_bar / sigma**2) / (n / sigma**2 + 1 / tau**2)
    shrinkage = 1 - map_est / x_bar  # how much MAP shrinks toward 0
    print(f'N(0, {tau}²)         {tau:>5.1f} {map_est:>8.3f} {shrinkage:>9.1%}')

print()
print('Smaller τ (tighter prior) → more shrinkage toward prior mean (0)')
print('Larger τ (weaker prior) → MAP approaches MLE')
print('This is exactly how Ridge regression works!')

---
## 9. Introduction to MCMC

When the posterior doesn't have a closed-form solution (which is most real-world cases), we use **Markov Chain Monte Carlo (MCMC)** to generate samples from the posterior.

**Core idea:** Instead of computing $P(\theta|D)$ analytically, generate a sequence of $\theta$ values whose long-run distribution converges to the posterior.

### Key MCMC Algorithms:
| Algorithm | Description | Pros | Cons |
|-----------|-------------|------|------|
| **Metropolis-Hastings** | General-purpose random walk | Simple, flexible | Can be slow |
| **Gibbs Sampling** | Sample each parameter conditionally | Efficient for conjugate models | Requires conditional distributions |
| **Hamiltonian MC (HMC)** | Uses gradient information | Fast convergence | Needs differentiable model |
| **NUTS** | Adaptive HMC (used by PyMC/Stan) | State-of-the-art | Complex implementation |

### 9.1 Metropolis-Hastings Algorithm from Scratch

Let's implement the simplest MCMC algorithm — Metropolis-Hastings — to sample from a Beta posterior.

In [ ]:
def metropolis_hastings(log_posterior, initial, n_samples=10000, proposal_sd=0.1):
    """
    Metropolis-Hastings MCMC sampler.
    
    Parameters:
    -----------
    log_posterior : function returning log-posterior density
    initial : starting value
    n_samples : number of samples to draw
    proposal_sd : std of Normal proposal distribution
    
    Returns:
    --------
    samples : array of MCMC samples
    accept_rate : fraction of proposals accepted
    """
    samples = np.zeros(n_samples)
    current = initial
    accepted = 0
    
    for i in range(n_samples):
        # Propose new value from Normal centered at current
        proposal = current + np.random.normal(0, proposal_sd)
        
        # Compute acceptance ratio (in log space for numerical stability)
        log_alpha = log_posterior(proposal) - log_posterior(current)
        
        # Accept or reject
        if np.log(np.random.uniform()) < log_alpha:
            current = proposal
            accepted += 1
        
        samples[i] = current
    
    return samples, accepted / n_samples

# Target: posterior for coin flip, 14 heads in 20 flips, Beta(2,2) prior
n, k = 20, 14

def log_posterior_coin(p):
    """Log posterior for coin bias parameter."""
    if p <= 0 or p >= 1:
        return -np.inf  # outside valid range
    log_prior = stats.beta.logpdf(p, 2, 2)      # Beta(2,2) prior
    log_lik = stats.binom.logpmf(k, n, p)        # Binomial likelihood
    return log_prior + log_lik

# Run MCMC
np.random.seed(42)
samples, accept_rate = metropolis_hastings(
    log_posterior_coin, initial=0.5, n_samples=20000, proposal_sd=0.1
)

# Discard burn-in (first 2000 samples)
burn_in = 2000
posterior_samples = samples[burn_in:]

print(f'MCMC completed: {len(samples)} samples, acceptance rate = {accept_rate:.1%}')
print(f'After burn-in ({burn_in}): {len(posterior_samples)} samples')
print(f'\nPosterior summary from MCMC:')
print(f'  Mean:   {posterior_samples.mean():.4f}')
print(f'  Median: {np.median(posterior_samples):.4f}')
print(f'  Std:    {posterior_samples.std():.4f}')
print(f'  95% CI: ({np.percentile(posterior_samples, 2.5):.4f}, {np.percentile(posterior_samples, 97.5):.4f})')

# Compare with exact Beta posterior
exact = stats.beta(2+k, 2+n-k)
print(f'\nExact posterior Beta({2+k}, {2+n-k}):')
print(f'  Mean:   {exact.mean():.4f}')
print(f'  Std:    {exact.std():.4f}')
ci = exact.interval(0.95)
print(f'  95% CI: ({ci[0]:.4f}, {ci[1]:.4f})')

### 9.2 MCMC Diagnostics

We need to verify that MCMC has converged and is producing reliable samples. Key diagnostic plots: **trace plot**, **autocorrelation**, and **posterior histogram**.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Trace plot — should look like a "hairy caterpillar"
axes[0,0].plot(samples, 'b-', alpha=0.5, lw=0.5)
axes[0,0].axhline(exact.mean(), color='red', ls='--', label='True posterior mean')
axes[0,0].axvline(burn_in, color='orange', ls='-', lw=2, label=f'Burn-in ({burn_in})')
axes[0,0].set_title('Trace Plot')
axes[0,0].set_xlabel('Iteration')
axes[0,0].set_ylabel('p')
axes[0,0].legend()

# 2. Posterior histogram vs exact
axes[0,1].hist(posterior_samples, bins=60, density=True, alpha=0.6, 
               color='steelblue', label='MCMC samples')
x_plot = np.linspace(0.2, 1.0, 200)
axes[0,1].plot(x_plot, exact.pdf(x_plot), 'r-', lw=2, label='Exact posterior')
axes[0,1].set_title('Posterior Distribution')
axes[0,1].set_xlabel('p')
axes[0,1].legend()

# 3. Autocorrelation — should decay quickly
max_lag = 100
acf = np.correlate(posterior_samples - posterior_samples.mean(), 
                    posterior_samples - posterior_samples.mean(), mode='full')
acf = acf[len(acf)//2:]  # take positive lags only
acf = acf / acf[0]       # normalize
axes[1,0].bar(range(max_lag), acf[:max_lag], color='steelblue', alpha=0.7)
axes[1,0].axhline(0, color='black', lw=0.5)
axes[1,0].axhline(1.96/np.sqrt(len(posterior_samples)), color='red', ls='--', alpha=0.5)
axes[1,0].axhline(-1.96/np.sqrt(len(posterior_samples)), color='red', ls='--', alpha=0.5)
axes[1,0].set_title('Autocorrelation')
axes[1,0].set_xlabel('Lag')
axes[1,0].set_ylabel('ACF')

# 4. Running mean — should stabilize
running_mean = np.cumsum(posterior_samples) / np.arange(1, len(posterior_samples)+1)
axes[1,1].plot(running_mean, 'b-', lw=1)
axes[1,1].axhline(exact.mean(), color='red', ls='--', label='True posterior mean')
axes[1,1].set_title('Running Mean (Convergence Check)')
axes[1,1].set_xlabel('Sample')
axes[1,1].set_ylabel('Cumulative mean')
axes[1,1].legend()

plt.suptitle('MCMC Diagnostics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Effective sample size (simple estimate)
# ESS ≈ n / (1 + 2*sum(autocorrelations))
ess_factor = 1 + 2 * np.sum(acf[1:50])  # use first 50 lags
ess = len(posterior_samples) / ess_factor
print(f'Effective Sample Size (ESS): {ess:.0f} out of {len(posterior_samples)}')
print(f'Efficiency: {ess/len(posterior_samples)*100:.1f}%')

---
## 10. Bayesian Analysis with PyMC

**PyMC** is Python's leading probabilistic programming library. It lets you define Bayesian models in a natural syntax and handles MCMC sampling automatically using the NUTS algorithm.

> **Note:** If PyMC is not installed, run `pip install pymc` first. We also show equivalent manual implementations.

### 10.1 PyMC: Coin Flip Model

Let's build our first PyMC model — the familiar coin flip example.

In [ ]:
try:
    import pymc as pm
    import arviz as az
    PYMC_AVAILABLE = True
    print(f'PyMC version: {pm.__version__}')
    print(f'ArviZ version: {az.__version__}')
except ImportError:
    PYMC_AVAILABLE = False
    print('PyMC not installed. Install with: pip install pymc')
    print('We will show the model code and use manual calculations instead.')

if PYMC_AVAILABLE:
    # Define the model
    with pm.Model() as coin_model:
        # Prior: p ~ Beta(2, 2)
        p = pm.Beta('p', alpha=2, beta=2)
        
        # Likelihood: observed 14 heads in 20 flips
        obs = pm.Binomial('obs', n=20, p=p, observed=14)
        
        # Sample from posterior using NUTS
        trace = pm.sample(2000, tune=1000, random_seed=42, 
                          return_inferencedata=True, progressbar=True)
    
    # Summary statistics
    print(az.summary(trace, var_names=['p']))
else:
    print()
    print('--- Model specification (PyMC syntax) ---')
    print('with pm.Model() as coin_model:')
    print("    p = pm.Beta('p', alpha=2, beta=2)             # Prior")
    print("    obs = pm.Binomial('obs', n=20, p=p, observed=14) # Likelihood")
    print('    trace = pm.sample(2000, tune=1000)             # MCMC')
    print()
    # Manual equivalent using conjugate solution
    exact = stats.beta(2+14, 2+20-14)
    print('Exact posterior (conjugate): Beta(16, 8)')
    print(f'  Mean: {exact.mean():.4f}')
    print(f'  Std:  {exact.std():.4f}')
    ci = exact.interval(0.95)
    print(f'  95% CI: ({ci[0]:.4f}, {ci[1]:.4f})')

### 10.2 PyMC: Bayesian Linear Regression

A more complex example — Bayesian linear regression with priors on slope, intercept, and noise.

In [ ]:
# Generate synthetic regression data
np.random.seed(42)
n_obs = 50
true_intercept = 2.0
true_slope = 3.5
true_sigma = 1.5

x_data = np.random.uniform(0, 5, n_obs)
y_data = true_intercept + true_slope * x_data + np.random.normal(0, true_sigma, n_obs)

if PYMC_AVAILABLE:
    with pm.Model() as regression_model:
        # Priors
        intercept = pm.Normal('intercept', mu=0, sigma=10)
        slope = pm.Normal('slope', mu=0, sigma=10)
        sigma = pm.HalfNormal('sigma', sigma=5)
        
        # Linear model
        mu = intercept + slope * x_data
        
        # Likelihood
        y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_data)
        
        # Sample
        trace_reg = pm.sample(2000, tune=1000, random_seed=42,
                              return_inferencedata=True)
    
    print(az.summary(trace_reg, var_names=['intercept', 'slope', 'sigma']))
else:
    print('--- Bayesian Linear Regression (PyMC syntax) ---')
    print('with pm.Model() as regression_model:')
    print("    intercept = pm.Normal('intercept', mu=0, sigma=10)")
    print("    slope = pm.Normal('slope', mu=0, sigma=10)")
    print("    sigma = pm.HalfNormal('sigma', sigma=5)")
    print('    mu = intercept + slope * x_data')
    print("    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_data)")
    print('    trace = pm.sample(2000, tune=1000)')
    print()
    
    # Compare with OLS
    from scipy.stats import linregress
    result = linregress(x_data, y_data)
    print('Frequentist OLS comparison:')
    print(f'  Intercept: {result.intercept:.4f} (true: {true_intercept})')
    print(f'  Slope:     {result.slope:.4f} (true: {true_slope})')
    residuals = y_data - (result.intercept + result.slope * x_data)
    print(f'  Sigma:     {residuals.std():.4f} (true: {true_sigma})')

# Plot data and regression line
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_data, y_data, alpha=0.6, label='Data')
x_line = np.linspace(0, 5, 100)
ax.plot(x_line, true_intercept + true_slope * x_line, 'r-', lw=2, 
        label=f'True: y = {true_intercept} + {true_slope}x')

# OLS fit
from scipy.stats import linregress
res = linregress(x_data, y_data)
ax.plot(x_line, res.intercept + res.slope * x_line, 'g--', lw=2,
        label=f'OLS: y = {res.intercept:.2f} + {res.slope:.2f}x')

ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Bayesian Linear Regression Data')
ax.legend()
plt.tight_layout()
plt.show()

---
## 11. Practical Example: Bayesian A/B Testing

A company runs an A/B test on their website checkout flow. They want to know if the new design (B) has a higher conversion rate than the current design (A). Unlike frequentist testing which only says "significant or not", Bayesian A/B testing tells us:
1. The probability that B is better than A
2. By how much B is likely better
3. The expected loss of choosing either option

### Step 1: Data Collection

In [ ]:
# A/B test data
# Group A (control): 1300 visitors, 120 converted
# Group B (new design): 1250 visitors, 145 converted

n_A, conv_A = 1300, 120
n_B, conv_B = 1250, 145

rate_A = conv_A / n_A
rate_B = conv_B / n_B

print('=== A/B Test: Checkout Flow Redesign ===')
print(f'Group A (control):    {conv_A}/{n_A} converted = {rate_A:.2%}')
print(f'Group B (new design): {conv_B}/{n_B} converted = {rate_B:.2%}')
print(f'Observed lift: {(rate_B - rate_A)/rate_A:.1%}')

# Frequentist comparison (for reference)
from scipy.stats import chi2_contingency
contingency = np.array([[conv_A, n_A - conv_A], [conv_B, n_B - conv_B]])
chi2, p_val, _, _ = chi2_contingency(contingency)
print(f'\nFrequentist chi-square test: p = {p_val:.4f}')

### Step 2: Bayesian Model

We use independent Beta-Binomial models for each group with a weakly informative prior Beta(1, 1).

In [ ]:
# Prior: Beta(1, 1) for both groups (uniform — no preference)
alpha_prior, beta_prior = 1, 1

# Posterior distributions
post_A = stats.beta(alpha_prior + conv_A, beta_prior + n_A - conv_A)
post_B = stats.beta(alpha_prior + conv_B, beta_prior + n_B - conv_B)

print('Posterior distributions:')
print(f'  p_A ~ Beta({alpha_prior + conv_A}, {beta_prior + n_A - conv_A})')
print(f'  p_B ~ Beta({alpha_prior + conv_B}, {beta_prior + n_B - conv_B})')
print(f'  Posterior mean A: {post_A.mean():.4f}')
print(f'  Posterior mean B: {post_B.mean():.4f}')

# Plot posteriors
x = np.linspace(0.05, 0.18, 500)
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, post_A.pdf(x), 'b-', lw=2, label=f'Group A: mean={post_A.mean():.4f}')
ax.fill_between(x, post_A.pdf(x), alpha=0.2, color='blue')
ax.plot(x, post_B.pdf(x), 'r-', lw=2, label=f'Group B: mean={post_B.mean():.4f}')
ax.fill_between(x, post_B.pdf(x), alpha=0.2, color='red')
ax.set_xlabel('Conversion Rate', fontsize=12)
ax.set_ylabel('Posterior Density', fontsize=12)
ax.set_title('Posterior Distributions for A/B Test', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

### Step 3: Monte Carlo Comparison

By drawing samples from both posteriors, we can directly compute the probability that B beats A and the distribution of the lift.

In [ ]:
# Draw 100,000 samples from each posterior
np.random.seed(42)
n_sim = 100000
samples_A = post_A.rvs(n_sim)
samples_B = post_B.rvs(n_sim)

# P(B > A) — probability that B is better
prob_B_better = np.mean(samples_B > samples_A)
print(f'P(B > A) = {prob_B_better:.4f} ({prob_B_better*100:.1f}%)')
print(f'P(A > B) = {1-prob_B_better:.4f} ({(1-prob_B_better)*100:.1f}%)')

# Distribution of the difference (lift)
diff = samples_B - samples_A
relative_lift = (samples_B - samples_A) / samples_A * 100  # percentage lift

print(f'\nAbsolute difference (p_B - p_A):')
print(f'  Mean:   {diff.mean():.4f} ({diff.mean()*100:.2f} pp)')
print(f'  Median: {np.median(diff):.4f}')
print(f'  95% CI: ({np.percentile(diff, 2.5):.4f}, {np.percentile(diff, 97.5):.4f})')

print(f'\nRelative lift ((p_B - p_A) / p_A):')
print(f'  Mean:   {relative_lift.mean():.1f}%')
print(f'  95% CI: ({np.percentile(relative_lift, 2.5):.1f}%, {np.percentile(relative_lift, 97.5):.1f}%)')

### Step 4: Expected Loss Analysis

Expected loss tells us: "If we choose B but A is actually better, how much do we lose on average?" This helps make a practical business decision.

In [ ]:
# Expected loss: what we lose if we pick the wrong variant
# Loss of choosing B = max(p_A - p_B, 0) — only lose when A is actually better
loss_choose_B = np.maximum(samples_A - samples_B, 0).mean()
loss_choose_A = np.maximum(samples_B - samples_A, 0).mean()

print('=== Expected Loss Analysis ===')
print(f'Expected loss if we choose A: {loss_choose_A:.5f} ({loss_choose_A*100:.3f} pp)')
print(f'Expected loss if we choose B: {loss_choose_B:.5f} ({loss_choose_B*100:.3f} pp)')
print(f'\nRecommendation: Choose {"B" if loss_choose_B < loss_choose_A else "A"}')
print(f'(lower expected loss)')

# Practical threshold: we only care about differences > 0.5 pp
threshold = 0.005  # 0.5 percentage points
prob_meaningful = np.mean(diff > threshold)
print(f'\nP(B beats A by more than {threshold*100:.1f} pp) = {prob_meaningful:.1%}')

### Step 5: Comprehensive Visualization Dashboard

Let's create a complete dashboard summarizing the Bayesian A/B test results.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Posterior distributions overlay
x = np.linspace(0.05, 0.18, 500)
axes[0,0].plot(x, post_A.pdf(x), 'b-', lw=2, label='Group A (control)')
axes[0,0].fill_between(x, post_A.pdf(x), alpha=0.15, color='blue')
axes[0,0].plot(x, post_B.pdf(x), 'r-', lw=2, label='Group B (new)')
axes[0,0].fill_between(x, post_B.pdf(x), alpha=0.15, color='red')
axes[0,0].set_title('Posterior Conversion Rates', fontsize=12)
axes[0,0].set_xlabel('Conversion Rate')
axes[0,0].legend()

# 2. Distribution of difference (B - A)
axes[0,1].hist(diff * 100, bins=80, density=True, alpha=0.6, color='purple')
axes[0,1].axvline(0, color='black', ls='-', lw=2, label='No difference')
axes[0,1].axvline(diff.mean() * 100, color='red', ls='--', lw=2, 
                   label=f'Mean = {diff.mean()*100:.2f} pp')
# Shade the "B is better" region
axes[0,1].axvspan(0, diff.max() * 100, alpha=0.05, color='green')
axes[0,1].set_title(f'Difference (B - A): P(B>A) = {prob_B_better:.1%}', fontsize=12)
axes[0,1].set_xlabel('Difference (percentage points)')
axes[0,1].legend()

# 3. Relative lift distribution
axes[1,0].hist(relative_lift, bins=80, density=True, alpha=0.6, color='teal')
axes[1,0].axvline(0, color='black', ls='-', lw=2)
axes[1,0].axvline(relative_lift.mean(), color='red', ls='--', lw=2,
                   label=f'Mean lift = {relative_lift.mean():.1f}%')
ci_lift = np.percentile(relative_lift, [2.5, 97.5])
axes[1,0].axvline(ci_lift[0], color='orange', ls=':', lw=1.5)
axes[1,0].axvline(ci_lift[1], color='orange', ls=':', lw=1.5,
                   label=f'95% CI: [{ci_lift[0]:.1f}%, {ci_lift[1]:.1f}%]')
axes[1,0].set_title('Relative Lift Distribution', fontsize=12)
axes[1,0].set_xlabel('Relative Lift (%)')
axes[1,0].legend()

# 4. Sequential updating — how P(B>A) evolves with more data
# Simulate accumulating data
np.random.seed(42)
check_ns = np.arange(50, n_A + 1, 50)
prob_B_wins = []
for n_check in check_ns:
    ratio = n_check / n_A
    n_a = int(n_A * ratio)
    n_b = int(n_B * ratio)
    c_a = int(conv_A * ratio)
    c_b = int(conv_B * ratio)
    
    samp_a = stats.beta(1 + c_a, 1 + n_a - c_a).rvs(10000)
    samp_b = stats.beta(1 + c_b, 1 + n_b - c_b).rvs(10000)
    prob_B_wins.append(np.mean(samp_b > samp_a))

axes[1,1].plot(check_ns, prob_B_wins, 'b-o', markersize=3)
axes[1,1].axhline(0.95, color='green', ls='--', alpha=0.5, label='95% threshold')
axes[1,1].axhline(0.5, color='gray', ls=':', alpha=0.5)
axes[1,1].set_title('P(B > A) Over Time', fontsize=12)
axes[1,1].set_xlabel('Total Visitors per Group')
axes[1,1].set_ylabel('P(B > A)')
axes[1,1].set_ylim(0, 1)
axes[1,1].legend()

plt.suptitle('Bayesian A/B Test Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 6: Business Summary Report

Let's generate a concise summary that a stakeholder could understand.

In [ ]:
print('=' * 60)
print('   BAYESIAN A/B TEST — EXECUTIVE SUMMARY')
print('=' * 60)
print()
print(f'Test: New checkout flow (B) vs Current (A)')
print(f'Sample: A = {n_A} visitors, B = {n_B} visitors')
print()
print('FINDINGS:')
print(f'  - Group A conversion rate: {rate_A:.2%}')
print(f'  - Group B conversion rate: {rate_B:.2%}')
print(f'  - Probability B is better: {prob_B_better:.1%}')
print(f'  - Expected lift: {relative_lift.mean():.1f}% '
      f'(95% CI: {ci_lift[0]:.1f}% to {ci_lift[1]:.1f}%)')
print(f'  - Expected loss if choosing B: {loss_choose_B*100:.3f} pp')
print(f'  - Expected loss if choosing A: {loss_choose_A*100:.3f} pp')
print()
if prob_B_better > 0.95:
    print('RECOMMENDATION: DEPLOY B')
    print('High confidence that the new design improves conversion.')
elif prob_B_better > 0.80:
    print('RECOMMENDATION: CONTINUE TESTING')
    print('B looks promising but we need more data for high confidence.')
else:
    print('RECOMMENDATION: KEEP A')
    print('Not enough evidence that B is meaningfully better.')
print()
print('Bayesian advantage: We can make direct probability statements')
print('and quantify the expected cost of each decision.')
print('=' * 60)

---
## 12. Summary

| Section | Key Concepts | Python Tools |
|---------|-------------|--------------|
| 1. Bayesian vs Frequentist | Parameters as random variables, degree of belief | `scipy.stats.beta` |
| 2. Bayes' Theorem | Posterior ∝ Likelihood × Prior, sequential updating | Manual computation |
| 3. Prior Distributions | Informative, weakly informative, non-informative, Jeffreys | `stats.beta.pdf()` |
| 4. Likelihood & Posterior | Grid approximation, numerical posterior | `np.linspace`, array ops |
| 5. Beta-Binomial | Conjugate pair for success/failure data | `stats.beta`, `stats.binom` |
| 6. Normal-Normal | Conjugate pair for mean estimation | `stats.norm`, precision weighting |
| 7. Credible vs Confidence | HPD interval, direct probability statements | Custom `hpd_interval()` |
| 8. MAP Estimation | Mode of posterior, regularization connection | `scipy.optimize` |
| 9. MCMC | Metropolis-Hastings, diagnostics, convergence | Custom implementation |
| 10. PyMC | Probabilistic programming, NUTS sampler | `pymc`, `arviz` |
| 11. Bayesian A/B Testing | P(B>A), expected loss, lift distribution | Monte Carlo simulation |

## Homework

### Task 1: Bayesian Coin Analysis
You flip a coin 30 times and get 21 heads.
1. Use a `Beta(5, 5)` prior (mild belief that the coin is fair).
2. Compute the posterior distribution analytically.
3. Calculate the posterior mean, mode, and 95% credible interval.
4. What is P(coin is biased toward heads, i.e., p > 0.5)?
5. Plot the prior, likelihood, and posterior on one figure.

### Task 2: Normal-Normal Estimation
A factory claims their widgets weigh 500g on average. You measure 20 widgets and get a mean of 497g (assume σ = 8g is known).
1. Set a prior of N(500, 5²) — trust the factory claim moderately.
2. Compute the posterior for μ using the Normal-Normal conjugate formula.
3. What is P(μ < 500 | data)? Is the factory claim plausible?
4. How would the result change with a vague prior N(500, 100²)?

### Task 3: Implement Metropolis-Hastings
Use the Metropolis-Hastings algorithm to sample from a Gamma(3, 1) distribution.
1. Implement the sampler with a Normal proposal (proposal_sd = 1.0).
2. Run 20,000 iterations and discard the first 5,000 as burn-in.
3. Create diagnostic plots: trace, histogram vs exact, and autocorrelation.
4. Experiment with different proposal_sd values (0.01, 0.1, 1.0, 10.0). What happens to the acceptance rate and mixing?

### Task 4: Bayesian A/B Test
An e-commerce company tests two email subject lines:
- Subject A: 500 sent, 45 opened
- Subject B: 480 sent, 52 opened

Use Bayesian A/B testing to:
1. Compute posterior distributions for open rates of A and B.
2. Estimate P(B > A) via Monte Carlo simulation.
3. Calculate the expected loss for choosing each variant.
4. Make a recommendation: which subject line should they use?
5. Create a visualization dashboard showing all results.

---
## Next Week Preview

**Week 13: Web Scraping with Python** — We'll learn how to collect data from the web using Python. Topics include HTTP basics, `requests` library, HTML parsing with `BeautifulSoup`, handling APIs, and ethical scraping practices.

---
*Applied Statistics with Python (2026) | Week 12 | Bayesian Statistics*